In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    repo = "jsoma/workshop-ai-images-video"
    url = f"https://github.com/{repo}/archive/refs/heads/main.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_repo.zip")
    with zipfile.ZipFile("_repo.zip") as zf:
        prefix = zf.namelist()[0]
        for m in zf.namelist():
            if m.startswith(prefix + "data/") and not m.endswith("/"):
                rel = m[len(prefix):]
                Path(rel).parent.mkdir(parents=True, exist_ok=True)
                Path(rel).write_bytes(zf.read(m))
    os.remove("_repo.zip")
    print("Done!")

load_dotenv()
DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# The full pipeline

## Screen time

Frames → classify → per-subject screen time with percentages. This is the deliverable you'd bring to an editor meeting.


**`recipes/screen-time.py`** — Classify pre-extracted debate frames → per-subject screen time summary.


In [ ]:
# Audit trail: every frame gets a row, not a vibe answer.
from pathlib import Path
import mimetypes, re
import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")

MODEL, SECONDS_PER_FRAME = "openai:gpt-4o-mini", 1.0

class FrameClassification(BaseModel):
    primary_subject: str = Field(description="Who is primarily on screen (name or description)")
    scene_type: str = Field(description="'close-up', 'wide shot', 'split screen', 'graphic', 'audience', 'other'")
    confidence: float = Field(ge=0.0, le=1.0)

# --- Step 1: Discover frames ---
frames_dir = DATA / "debate"
frames = sorted(p for p in frames_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"Found {len(frames)} frames")

# --- Step 2: Classify each frame ---
agent = Agent(MODEL, output_type=FrameClassification)
rows = []
for fpath in frames:
    num = int(m.group(1)) if (m := re.search(r"(\d+)", fpath.stem)) else 0
    mime, _ = mimetypes.guess_type(str(fpath))
    r = agent.run_sync([
        "This is a frame from a political debate. Identify who is on screen.",
        BinaryContent(data=fpath.read_bytes(), media_type=mime or "image/jpeg"),
    ])
    rows.append({"filename": fpath.name, "frame_number": num, "timestamp_sec": num * SECONDS_PER_FRAME,
                 "primary_subject": r.output.primary_subject, "scene_type": r.output.scene_type,
                 "confidence": r.output.confidence})
print(f"Classified {len(rows)} frames")

# --- Step 3: Build summary ---
df = pd.DataFrame(rows).sort_values("frame_number").reset_index(drop=True)
summary = (
    df.groupby("primary_subject")
    .agg(frames=("primary_subject", "count"),
         total_seconds=("timestamp_sec", lambda x: len(x) * SECONDS_PER_FRAME),
         avg_confidence=("confidence", "mean"))
    .sort_values("total_seconds", ascending=False)
)
summary["pct_of_total"] = (summary["total_seconds"] / summary["total_seconds"].sum() * 100).round(1)
print(summary)

df


## Cost

Before you run a big batch: how much will it cost? Images × tokens × price = receipt.


**`recipes/cost.py`** — Estimate API cost for a batch of images. No API key needed.


In [ ]:
# USD per million tokens (Feb 2026)
PRICE_TABLE = {
    "gpt-4o-mini":       (0.15, 0.60),
    "gpt-4o":            (2.50, 10.00),
    "gpt-5-nano":        (0.05, 0.20),
    "gemini-2.5-flash":  (0.10, 0.40),
    "gemini-2.5-pro":    (1.25, 5.00),
    "claude-3-5-haiku":  (0.80, 4.00),
    "claude-3-5-sonnet": (3.00, 15.00),
    "ollama":            (0.00, 0.00),
}

NUM_IMAGES = 500
MODEL = "gpt-4o-mini"
INPUT_TOKENS_PER_IMAGE = 1000
OUTPUT_TOKENS_PER_IMAGE = 200

# --- Calculate ---
input_price, output_price = PRICE_TABLE[MODEL]
total_input = NUM_IMAGES * INPUT_TOKENS_PER_IMAGE
total_output = NUM_IMAGES * OUTPUT_TOKENS_PER_IMAGE
input_cost = (total_input / 1_000_000) * input_price
output_cost = (total_output / 1_000_000) * output_price
total_cost = input_cost + output_cost

print(f"Model:       {MODEL}")
print(f"Images:      {NUM_IMAGES:,}")
print(f"Input cost:  ${input_cost:.4f}  ({total_input:,} tokens @ ${input_price}/M)")
print(f"Output cost: ${output_cost:.4f}  ({total_output:,} tokens @ ${output_price}/M)")
print(f"TOTAL:       ${total_cost:.4f}")
